# 🦓 Dein erstes YOLO-Training – African Wildlife

<a href="https://colab.research.google.com/github/rdf-sul/YOLO_Training/blob/main/notebooks/01_erstes_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="In Colab öffnen"/></a>

In diesem Notebook trainierst du ein **YOLO26**-Modell, das **Büffel, Elefanten, Nashörner und Zebras** erkennt.
Du brauchst nichts zu labeln – der Datensatz ist fertig und wird automatisch heruntergeladen.

| Schritt | Was passiert | Dauer |
|---|---|---|
| 1 | GPU prüfen | 1 min |
| 2 | Ultralytics mit **uv** installieren | 1 min |
| 3 | Vorher-Test: Was erkennt ein Modell *ohne* unser Training? | 2 min |
| 4 | **Training starten** | ca. 15–20 min |
| 5 | Ergebnisse ansehen (Loss, mAP, Confusion Matrix) | 10 min |
| 6 | Validierung auf den Testbildern | 2 min |
| 7 | Eigene Bilder testen | 5 min |
| 8 | Modell sichern | 1 min |
| 🧪 | Experimente | optional |

📖 Begleitende Theorie: [https://rdf-sul.github.io/YOLO_Training](https://rdf-sul.github.io/YOLO_Training)

> **Wichtig:** Stelle zuerst die GPU ein: Menü **Laufzeit → Laufzeittyp ändern → T4 GPU → Speichern**.
> Führe die Zellen dann **der Reihe nach** mit ▶ (oder `Umschalt + Enter`) aus.

## ⚙️ Einstellungen

Hier stehen alle wichtigen Werte an einem Ort. Für das erste Training musst du nichts ändern.

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  EINSTELLUNGEN – hier kannst du später experimentieren   ║
# ╚══════════════════════════════════════════════════════════╝
MODELL   = "yolo26n.pt"            # Startmodell: yolo26n.pt, yolo26s.pt, ... (vortrainiert)
DATEN    = "african-wildlife.yaml" # Datensatz (wird automatisch geladen)
EPOCHEN  = 30                      # Anzahl Epochen
BILDGR   = 640                     # imgsz: Bildgröße in Pixeln
BATCH    = 16                      # Bilder pro Batch
PATIENCE = 10                      # Early Stopping nach N Epochen ohne Verbesserung
NAME     = "wildlife_n_30ep"       # Name des Ergebnisordners

SCHNELLTEST = False                # True = nur 1 Epoche mit 10 % der Daten (zum Ausprobieren)

## 1 · GPU prüfen

Training auf einer Grafikkarte (GPU) ist etwa **20–50× schneller** als auf dem Prozessor (CPU).
Wenn unten `Tesla T4` steht, ist alles richtig eingestellt.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "⚠️ Keine GPU gefunden – bitte Laufzeittyp auf T4 GPU stellen!"

## 2 · Ultralytics mit uv installieren

Wir verwenden **uv** statt pip – es ist deutlich schneller. In Colab installieren wir direkt in die vorhandene
Python-Umgebung (`--system`). Zu Hause würdest du `uv add ultralytics` in einem Projekt verwenden.

In [ ]:
if IN_COLAB:
    !pip install -q uv
    !uv pip install --system -q ultralytics

import ultralytics
ultralytics.checks()

## 3 · Vorher-Test: Was kann das Modell *ohne* unser Training?

Das Startmodell `yolo26n.pt` wurde auf dem **COCO-Datensatz** (80 Alltagsklassen) trainiert.
Darin kommen **Elefanten und Zebras** vor – aber **keine Büffel und keine Nashörner**.

🤔 **Vermutung notieren:** Was wird das Modell bei einem Nashorn-Bild ausgeben?

In [ ]:
from pathlib import Path
import random
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset

info = check_det_dataset(DATEN)          # lädt den Datensatz herunter (ca. 100 MB), falls noch nicht vorhanden
DATEN_ORDNER = Path(info["path"])
print("Datensatz liegt in:", DATEN_ORDNER)
print("Klassen:", info["names"])
for teil in ["train", "val", "test"]:
    print(f"{teil:5s}: {len(list((DATEN_ORDNER / 'images' / teil).glob('*.jpg')))} Bilder")

In [ ]:
import matplotlib.pyplot as plt

random.seed(3)
testbilder = sorted((DATEN_ORDNER / "images" / "test").glob("*.jpg"))
auswahl = random.sample(testbilder, 4)

coco_modell = YOLO(MODELL)
ergebnisse = coco_modell(auswahl, verbose=False)

fig, achsen = plt.subplots(1, 4, figsize=(18, 5))
for ax, erg in zip(achsen, ergebnisse):
    ax.imshow(erg.plot()[:, :, ::-1])   # BGR -> RGB
    ax.set_title(Path(erg.path).name)
    ax.axis("off")
plt.suptitle("Vorher: COCO-Modell OHNE Wildlife-Training", fontsize=14)
plt.show()

**Beobachtung:** Was wurde erkannt? Welche Tiere bekommen falsche Namen? (Typisch: Nashörner werden zu *elephant*, Büffel zu *cow* – das Modell nimmt die ähnlichste Klasse, die es kennt.)
👉 Genau dafür trainieren wir jetzt nach.

## 4 · Training starten 🚀

Jetzt passiert das eigentliche Training. Achte auf die Ausgabe:

- **Epoch** `5/30` – aktuelle Epoche
- **box_loss, cls_loss, l1_loss** – die Fehlerwerte. Sie sollten **sinken**.
- Nach jeder Epoche: **P, R, mAP50, mAP50-95** auf den Validierungsbildern. Sie sollten **steigen**.

⏳ **Während das Training läuft:** Lies auf der Lernseite die Einheit **„Ergebnisse verstehen“** und bearbeite die Experimente dort.

In [ ]:
model = YOLO(MODELL)

extra = dict(epochs=1, fraction=0.1) if SCHNELLTEST else dict(epochs=EPOCHEN)
ergebnis = model.train(
    data=DATEN,
    imgsz=BILDGR,
    batch=BATCH,
    patience=PATIENCE,
    name=NAME,
    exist_ok=True,
    cache=True,          # Bilder im RAM halten -> schneller
    plots=True,
    **extra,
)
ERGEBNIS_ORDNER = Path(model.trainer.save_dir)
print("\n✅ Fertig! Ergebnisse liegen in:", ERGEBNIS_ORDNER)

## 5 · Ergebnisse ansehen

Ultralytics legt im Ergebnisordner automatisch viele Dateien an. Schau sie dir an und vergleiche mit der
Lernseite **„Ergebnisse verstehen“**.

In [ ]:
for f in sorted(ERGEBNIS_ORDNER.iterdir()):
    print(f"{'📁' if f.is_dir() else '📄'} {f.name}")

In [ ]:
from IPython.display import Image, display

def zeige(name, breite=900):
    pfad = ERGEBNIS_ORDNER / name
    if pfad.exists():
        print("▶", name)
        display(Image(filename=str(pfad), width=breite))
    else:
        print("(nicht vorhanden:", name, ")")

zeige("results.png")                        # Loss- und Metrik-Kurven über die Epochen

**Fragen zu `results.png`:**
1. Sinken alle drei `train/…_loss`-Kurven? Sinken auch die `val/…_loss`-Kurven?
2. Ab welcher Epoche verbessert sich `mAP50-95` kaum noch?
3. Gibt es Anzeichen von **Overfitting** (val-Loss steigt wieder)?

In [ ]:
zeige("confusion_matrix_normalized.png", 650)

**Fragen zur Confusion Matrix:**
1. Welche Klasse wird am besten erkannt (höchster Wert auf der Diagonalen)?
2. Welche Tiere werden am häufigsten **verwechselt**?
3. Was bedeutet die Spalte bzw. Zeile **background**?

In [ ]:
zeige("BoxPR_curve.png", 650)
zeige("labels.jpg", 650)

In [ ]:
# Vergleich: Labels (links) und Vorhersagen (rechts) auf Validierungsbildern
fig, achsen = plt.subplots(1, 2, figsize=(18, 9))
for ax, name, titel in zip(achsen, ["val_batch0_labels.jpg", "val_batch0_pred.jpg"], ["Labels (Ground Truth)", "Vorhersagen"]):
    pfad = ERGEBNIS_ORDNER / name
    if pfad.exists():
        ax.imshow(plt.imread(str(pfad)))
    ax.set_title(titel, fontsize=14)
    ax.axis("off")
plt.show()

## 6 · Validierung auf den Testbildern

Bisher wurde mit den **Validierungsbildern** gemessen. Jetzt kommt die „Abschlussprüfung“ auf den **Testbildern**,
die das Modell noch nie gesehen hat.

In [ ]:
bestes = YOLO(ERGEBNIS_ORDNER / "weights" / "best.pt")
m = bestes.val(data=DATEN, split="test", imgsz=BILDGR, plots=False)

print(f"\n{'Klasse':10s} {'mAP50':>7s} {'mAP50-95':>9s}")
for i, name in bestes.names.items():
    print(f"{name:10s} {m.box.ap50[i]:7.3f} {m.box.maps[i]:9.3f}")
print(f"{'GESAMT':10s} {m.box.map50:7.3f} {m.box.map:9.3f}")

## 7 · Vorhersagen ansehen – und eigene Bilder testen

In [ ]:
random.seed(7)
auswahl = random.sample(testbilder, 6)
ergebnisse = bestes(auswahl, conf=0.25, verbose=False)

fig, achsen = plt.subplots(2, 3, figsize=(18, 10))
for ax, erg in zip(achsen.flat, ergebnisse):
    ax.imshow(erg.plot()[:, :, ::-1])
    ax.set_title(f"{Path(erg.path).name}: {len(erg.boxes)} Tier(e)")
    ax.axis("off")
plt.suptitle("Nachher: unser trainiertes Modell", fontsize=14)
plt.show()

**Eigenes Bild testen:** Suche im Internet ein Foto von einem Zebra, Elefanten, Büffel oder Nashorn (oder einem ganz
anderen Tier – was passiert dann?) und lade es hoch.

In [ ]:
if IN_COLAB:
    from google.colab import files
    hochgeladen = files.upload()
    for dateiname in hochgeladen:
        erg = bestes(dateiname, conf=0.25, verbose=False)[0]
        plt.figure(figsize=(10, 7))
        plt.imshow(erg.plot()[:, :, ::-1])
        plt.axis("off")
        plt.show()
else:
    print("Upload nur in Colab verfügbar.")

## 8 · Modell sichern 💾

Colab löscht alle Dateien, wenn die Sitzung endet! Lade dein Modell herunter – du brauchst es für den
**Live-Test mit Ultralytics Solutions** auf deinem Laptop.

In [ ]:
import shutil
ziel = Path(f"{NAME}_best.pt")
shutil.copy(ERGEBNIS_ORDNER / "weights" / "best.pt", ziel)
shutil.make_archive(NAME, "zip", ERGEBNIS_ORDNER)      # kompletter Ergebnisordner als ZIP
print("Gesichert:", ziel, "und", f"{NAME}.zip")

if IN_COLAB:
    from google.colab import files
    files.download(str(ziel))
    files.download(f"{NAME}.zip")

*Optional:* In Google Drive speichern (bleibt dauerhaft erhalten):

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_ziel = Path("/content/drive/MyDrive/YOLO_Training")
    drive_ziel.mkdir(parents=True, exist_ok=True)
    shutil.copy(ziel, drive_ziel / ziel.name)
    shutil.copy(f"{NAME}.zip", drive_ziel / f"{NAME}.zip")
    print("In Drive gespeichert:", drive_ziel)

---
# 🧪 Experimente

Jetzt wirst du zur Forscherin bzw. zum Forscher. Ändere **immer nur einen Parameter** gegenüber dem ersten Training
und vergleiche die Ergebnisse. Teilt die Experimente in der Klasse auf und tragt die Ergebnisse zusammen!

| Nr. | Experiment | Änderung | Frage |
|---|---|---|---|
| A | Kurz trainiert | `epochs=10` | Wie viel mAP kosten 20 Epochen weniger? |
| B | Größeres Modell | `yolo26s.pt` | Lohnt sich die längere Trainingszeit? |
| C | Kleine Bilder | `imgsz=320` | Wie stark leidet die Genauigkeit, wie viel schneller ist es? |
| D | Weniger Daten | `fraction=0.25` | Was passiert mit nur 25 % der Trainingsbilder? |
| E | Ohne Vorwissen | `yolo26n.yaml` (von Null) | Wie wichtig ist Transfer Learning? |
| F | Ältere Generation | `yolo11n.pt` | Ist YOLO26 wirklich besser als YOLO11? |

**Ergebnistabelle** (abschreiben und ausfüllen):

| Nr. | mAP50 (val) | mAP50-95 (val) | Trainingszeit | Beobachtung |
|---|---|---|---|---|
| Basis | | | | |
| A | | | | |
| … | | | | |

In [ ]:
# ▶ Experiment: Trage hier deine Änderung ein und führe die Zelle aus
EXPERIMENT = dict(
    model="yolo26n.pt",   # B: "yolo26s.pt" | E: "yolo26n.yaml" | F: "yolo11n.pt"
    epochs=10,            # A: 10
    imgsz=640,            # C: 320
    fraction=1.0,         # D: 0.25
    name="exp_A_10ep",    # eindeutiger Name!
)

import time
start = time.time()
exp_model = YOLO(EXPERIMENT.pop("model"))
exp_model.train(data=DATEN, batch=BATCH, patience=PATIENCE, exist_ok=True, cache=True, **EXPERIMENT)
dauer = (time.time() - start) / 60
mm = exp_model.trainer.metrics
print(f"\n⏱️ Dauer: {dauer:.1f} min | mAP50: {mm['metrics/mAP50(B)']:.3f} | mAP50-95: {mm['metrics/mAP50-95(B)']:.3f}")

### Ausblick

Gut gemacht! 🎉 Du hast ein eigenes Objekterkennungsmodell trainiert und bewertet.

**Nächster Schritt:** Lade `best.pt` herunter und teste es auf deinem Laptop live mit **Ultralytics Solutions**
(Seite „Modell testen mit Solutions“ auf der Lernseite).